In [1]:
from jax import jit
import jax.numpy as jnp
import numpy as np

@jit
def f(x, y):
    print("Running f():")
    print(f"  {x=}")
    print(f"  {y=}")
    result = jnp.dot(x + 1, y + 1)
    print(f"  {result=}")
    return result

x = np.random.randn(3, 4)
y = np.random.randn(4)
f(x, y)

Running f():
  x=JitTracer(float32[3,4])
  y=JitTracer(float32[4])
  result=JitTracer(float32[3])


Array([7.5309086, 2.7863917, 3.5197875], dtype=float32)

In [2]:
x2 = np.random.randn(3, 4)
y2 = np.random.randn(4)
f(x2, y2)

Array([1.7281599, 3.586465 , 0.8406482], dtype=float32)

In [3]:
x2 = np.random.randn(4, 4)
y2 = np.random.randn(4)
f(x2, y2)

Running f():
  x=JitTracer(float32[4,4])
  y=JitTracer(float32[4])
  result=JitTracer(float32[4])


Array([1.5376642, 4.3164253, 5.865443 , 5.6281123], dtype=float32)

In [4]:
from jax import make_jaxpr

def f(x, y):
    return jnp.dot(x + 1, y + 1)

make_jaxpr(f)(x, y)

{ lambda ; a:f32[3,4] b:f32[4]. let
    c:f32[3,4] = add a 1.0:f32[]
    d:f32[4] = add b 1.0:f32[]
    e:f32[3] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] c d
  in (e,) }

In [9]:
@jit
def f(x):
    return x.reshape(jnp.array(x.shape).prod())

x = jnp.ones((2, 3))
try:
    f(x)
except Exception as ex:
    print(ex)

Shapes must be 1D sequences of concrete values of integer type, got [JitTracer(int32[])].
If using `jit`, try using `static_argnums` or applying `jit` to smaller subfunctions.
The error occurred while tracing the function f at /tmp/ipykernel_2673165/3683230533.py:1 for jit. This value became a tracer due to JAX operations on these lines:

  operation a:i32[] = reduce_prod[axes=(0,)] b
    from line /tmp/ipykernel_2673165/3683230533.py:3:21 (f)


In [10]:
jnp.array(x.shape)

Array([2, 3], dtype=int32)

In [11]:
jnp.array(x.shape).prod()

Array(6, dtype=int32)

In [12]:
@jit
def f(x):
    print(f"{x=}")
    print(f"{x.shape=}")
    print(f"{jnp.array(x.shape).prod()=}")

f(x)

x=JitTracer(float32[2,3])
x.shape=(2, 3)
jnp.array(x.shape).prod()=JitTracer(int32[])


In [16]:
@jit
def f(x):
    print(f"{x.shape=}")
    print(f"{np.prod(x.shape)=}")
    print(f"{x.reshape((np.prod(x.shape),))=}")
    return x.reshape((np.prod(x.shape),))

f(x)

x.shape=(2, 3)
np.prod(x.shape)=np.int64(6)
x.reshape((np.prod(x.shape),))=JitTracer(float32[6])


Array([1., 1., 1., 1., 1., 1.], dtype=float32)